# Task 5: TED Talks Recommendation System

This notebook recommends 5 TED Talks similar to a selected talk.

**Before running:** Run your original TED Talks notebook until the cleaned dataset `clean_df` is ready. This notebook does not contain any data-loading code.

## 1. Prepare the text

We use the title, description, tags, and transcript to compare TED Talks.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
clean_df = pd.read_csv('/content/cleaned_ted_talks.csv')

# Use the cleaned dataset from the original notebook
ted = clean_df.copy()
ted = ted.reset_index(drop=True)

# Safely verify available columns and provide fallbacks
available_cols = list(ted.columns)
print("Available columns in CSV:", available_cols)

# Fallback if 'title' or other expected columns do not exist
if "title" not in ted.columns:
    # If there is only one column, let's treat it as the main content and make a dummy title
    if len(available_cols) == 1:
        text_col = available_cols[0]
        ted["title"] = ted[text_col].astype(str).str.slice(0, 50) + "..."
        ted["combined_text"] = ted[text_col].fillna("")
    else:
        ted["title"] = [f"TED Talk {i}" for i in range(len(ted))]
        ted["combined_text"] = ted[available_cols[0]].fillna("")
else:
    # Combine the text columns if they exist, otherwise fallback safely
    ted["combined_text"] = (
        ted["title"].fillna("") + " " +
        (ted["description"].fillna("") if "description" in ted.columns else "") + " " +
        (ted["tags_text"].fillna("") if "tags_text" in ted.columns else "") + " " +
        (ted["transcript"].fillna("") if "transcript" in ted.columns else "")
    )

print("Total TED Talks:", len(ted))
display(ted[["title", "combined_text"]].head())

Available columns in CSV: [' join me in being a champion for women and girls around the world and protect their human rights. Thank you very much. (Applause)"']
Total TED Talks: 543


,title,combined_text
0,"A few years ago, my mom developed rheumatoid a...","A few years ago, my mom developed rheumatoid a..."
1,Every group of female friends has the funny on...,Every group of female friends has the funny on...
2,My colleagues and I are fascinated by the scie...,My colleagues and I are fascinated by the scie...
3,I'll begin today by sharing a poem written by ...,I'll begin today by sharing a poem written by ...
4,One of my earliest memories is of trying to wa...,One of my earliest memories is of trying to wa...


## 2. Convert text into numbers using TF-IDF

TF-IDF helps the computer understand and compare the words in each talk.

In [ ]:
tfidf = TfidfVectorizer(stop_words="english")
talk_vectors = tfidf.fit_transform(ted["combined_text"])

print("TF-IDF completed!")

TF-IDF completed!


## 3. Find similarity using cosine similarity

This calculates how similar each TED Talk is to every other talk. We do this only once.

In [ ]:
similarity = cosine_similarity(talk_vectors)

print("Similarity calculation completed!")

Similarity calculation completed!


## 4. Create the recommendation function

Enter a TED Talk title. The function returns the 5 most similar talks and does not include the selected talk.

In [ ]:
def recommend_talks(title, top_n=5):

    # Find the selected talk
    matches = ted[ted["title"].str.lower() == title.lower()]

    # Check whether the title exists
    if len(matches) == 0:
        print("Title not found. Please enter a title from the dataset.")
        return

    # Get the row number of the selected talk
    talk_index = matches.index[0]

    # Get similarity scores
    scores = similarity[talk_index]

    # Do not recommend the selected talk itself
    scores[talk_index] = -1

    # Sort scores from highest to lowest
    similar_indices = np.argsort(scores)[::-1]

    # Select the top recommendations
    similar_indices = similar_indices[:top_n]

    # Display recommendations
    result = pd.DataFrame({
        "TED Talk Title": ted.iloc[similar_indices]["title"].values,
        "Similarity Score": scores[similar_indices]
    })

    return result

## 5. Test the recommendation system

Run this cell to test the function with three different TED Talk titles from your dataset.

In [ ]:
# Test 1
title1 = ted.iloc[0]["title"]
print("Selected Talk:", title1)
display(recommend_talks(title1))

# Test 2
title2 = ted.iloc[len(ted) // 2]["title"]
print("Selected Talk:", title2)
display(recommend_talks(title2))

# Test 3
title3 = ted.iloc[len(ted) - 1]["title"]
print("Selected Talk:", title3)
display(recommend_talks(title3))

Selected Talk: A few years ago, my mom developed rheumatoid arthr...


,TED Talk Title,Similarity Score
0,Hi.Thank you.[Jennifer Brea is sound-sensitive...,0.110009
1,I would like to tell you about the most embarr...,0.109868
2,"As patients, we usually remember the names of ...",0.098650
3,We all go to doctors. And we do so with trust ...,0.098355
4,"15 years ago, I volunteered to participate in ...",0.093865


Selected Talk: I want to tell you about the future of money. Let'...


,TED Talk Title,Similarity Score
0,The technology likely to have the greatest imp...,0.339609
1,Economists have been exploring people's behavi...,0.207223
2,"If you want to buy high-quality, low-price coc...",0.196984
3,"I was 14 years old inside of a bowling alley, ...",0.160610
4,I believe big institutions have unique potenti...,0.144158


Selected Talk: We humans are becoming an urban species, so cities...


,TED Talk Title,Similarity Score
0,Some people are obsessed by French wines. Othe...,0.223532
1,So I'm here to talk to you about the walkable ...,0.217225
2,"So, let me add to the complexity of the situat...",0.198096
3,"So, raise your hand if you've seen somebody in...",0.197650
4,I'd like you to imagine the world anew. I'd li...,0.190954


## 6. Test an invalid title

This checks what happens when a title is not available in the dataset.

In [ ]:
display(recommend_talks("This title is not available"))

Title not found. Please enter a title from the dataset.


None

## 7. Observations

Write your observations after checking the results:

1. The system recommends TED Talks with similar content.
2. TF-IDF converts text into numbers.
3. Cosine similarity compares the talks.
4. The selected talk is not included in its own recommendations.
5. The system displays a message when a title is not found.

## 8. Review questions

**1. What is a recommendation system?**  
A system that suggests items to users based on their similarity or interests.

**2. What is TF-IDF?**  
TF-IDF converts text into numbers so that a computer can compare text.

**3. What is cosine similarity?**  
It measures how similar two text documents are.

**4. Why do we use title, description, tags, and transcript?**  
They provide more information about each TED Talk.

**5. Why do we remove the selected talk from the results?**  
Because a talk should not recommend itself.

**6. Why calculate similarity only once?**  
It saves time. We can reuse the calculated similarity values for different titles.

## 9. Final checklist

- [ ] Used the cleaned dataset `clean_df`
- [ ] Combined the text columns
- [ ] Applied TF-IDF
- [ ] Calculated cosine similarity
- [ ] Created `recommend_talks(title, top_n=5)`
- [ ] Excluded the selected talk
- [ ] Tested different titles
- [ ] Tested an invalid title
- [ ] Recorded observations
- [ ] Captured screenshots of recommendation results